# 📊 مصورسازی داده در Climatology Engine

این نوت‌بوک روش‌های مختلف مصورسازی نتایج را معرفی می‌کند.

**مواردی که یاد می‌گیرید:**
- بارگذاری نتایج از فایل Zarr
- رسم سری‌های زمانی (Time Series)
- رسم نقشه‌های مکانی توزیع‌ها
- رسم هیستوگرام و منحنی‌های توزیع
- رسم نمودارهای مقایسه‌ای
- استفاده از Cartopy برای نقشه‌های حرفه‌ای
- ذخیره نمودارها در فرمت‌های مختلف

---

## 📐 مقدمه

مصورسازی داده یکی از مهم‌ترین بخش‌های تحلیل داده است. در این نوت‌بوک با کتابخانه‌های زیر کار می‌کنیم:

- `matplotlib`: پایه‌ترین کتابخانه مصورسازی
- `seaborn`: مصورسازی آماری پیشرفته
- `cartopy`: نقشه‌های جغرافیایی (اختیاری)
- `plotly`: نمودارهای تعاملی (اختیاری)

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# تنظیمات نمایش
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.2)
%matplotlib inline

print('✅ کتابخانه‌ها بارگذاری شدند.')

In [ ]:
# بررسی وجود Cartopy
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    cartopy_available = True
    print('✅ Cartopy موجود است.')
except ImportError:
    cartopy_available = False
    print('⚠️ Cartopy نصب نیست. برای نصب: pip install cartopy')

# بررسی وجود Plotly
try:
    import plotly.express as px
    import plotly.graph_objects as go
    plotly_available = True
    print('✅ Plotly موجود است.')
except ImportError:
    plotly_available = False
    print('⚠️ Plotly نصب نیست. برای نصب: pip install plotly')

In [ ]:
# بارگذاری داده نمونه و برازش توزیع‌ها (برای ایجاد داده برای مصورسازی)
from core.engine.plugin_loader import load_plugins

plugins = load_plugins()
distributions = {dist.name: dist for dist in plugins.values()}

sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values
data_year = data[:365, 1]  # tmean

# برازش همه توزیع‌ها
results = {}
for name, dist in distributions.items():
    try:
        results[name] = dist.fit(data_year)
    except:
        pass

print(f"✅ {len(results)} توزیع برازش شد.")

In [ ]:
# ============================================================================
# ۱. رسم سری زمانی (Time Series)
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(data_year, color='blue', alpha=0.7, linewidth=1.5, label='داده روزانه')
ax.axhline(np.mean(data_year), color='red', linestyle='--', linewidth=2, label=f'میانگین = {np.mean(data_year):.2f}°C')

ax.set_xlabel('روز سال', fontsize=12)
ax.set_ylabel('دما (°C)', fontsize=12)
ax.set_title('سری زمانی دمای میانگین روزانه', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# ۲. هیستوگرام و منحنی‌های توزیع برازش شده
# ============================================================================

fig, ax = plt.subplots(figsize=(12, 7))

# هیستوگرام داده
ax.hist(data_year, bins=30, density=True, alpha=0.4, color='gray', edgecolor='black', label='داده')

# منحنی‌های توزیع
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
x = np.linspace(min(data_year), max(data_year), 500)

for i, (name, res) in enumerate(results.items()):
    dist = distributions[name]
    if hasattr(dist, 'pdf'):
        try:
            params = {p: res[p] for p in dist.params if p in res}
            pdf_vals = dist.pdf(x, params)
            ax.plot(x, pdf_vals, color=colors[i % len(colors)], 
                    linewidth=2.5, label=f'{name} (AICc={res["aicc"]:.1f})')
        except:
            pass

ax.set_xlabel('دما (°C)', fontsize=12)
ax.set_ylabel('چگالی احتمال', fontsize=12)
ax.set_title('هیستوگرام و منحنی‌های توزیع برازش شده', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# ۳. نمودار QQ-Plot (مقایسه توزیع نرمال)
# ============================================================================

from scipy import stats

fig, ax = plt.subplots(figsize=(8, 8))

# QQ-Plot برای توزیع نرمال
stats.probplot(data_year, dist='norm', plot=ax)

ax.set_title('QQ-Plot (مقایسه با توزیع نرمال)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# ۴. نقشه مکانی توزیع‌ها (با استفاده از داده‌های مصنوعی)
# ============================================================================

# تولید داده‌های مصنوعی با مختصات
np.random.seed(42)
n_stations = 50
lats = np.random.uniform(25, 40, n_stations)
lons = np.random.uniform(44, 64, n_stations)
dist_codes = np.random.choice([0, 1, 2, 3, 4], n_stations)
dist_names = ['Normal', 'Skew', 'GEV', 'Bimodal', 'Pearson']

df_map = pd.DataFrame({
    'lat': lats,
    'lon': lons,
    'distribution': [dist_names[c] for c in dist_codes],
    'code': dist_codes
})

print(f"📊 {len(df_map)} نقطه با مختصات تولید شد.")

In [ ]:
# رسم نقشه با scatter (ساده)
fig, ax = plt.subplots(figsize=(12, 10))

colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

for i, dist_name in enumerate(dist_names):
    mask = df_map['distribution'] == dist_name
    if mask.any():
        ax.scatter(df_map.loc[mask, 'lon'], df_map.loc[mask, 'lat'], 
                   c=colors[i], s=100, alpha=0.7, edgecolor='black', linewidth=1,
                   label=dist_name)

ax.set_xlabel('طول جغرافیایی', fontsize=12)
ax.set_ylabel('عرض جغرافیایی', fontsize=12)
ax.set_title('نقشه توزیع بهترین مدل در نقاط مختلف', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# رسم نقشه با Cartopy (اگر موجود باشد)
if cartopy_available:
    fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={'projection': ccrs.PlateCarree()})
    
    # اضافه کردن ویژگی‌های نقشه
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':')
    ax.add_feature(cfeature.LAKES, facecolor='lightblue', alpha=0.5)
    ax.add_feature(cfeature.RIVERS, linewidth=0.3)
    
    # رسم نقاط
    for i, dist_name in enumerate(dist_names):
        mask = df_map['distribution'] == dist_name
        if mask.any():
            ax.scatter(df_map.loc[mask, 'lon'], df_map.loc[mask, 'lat'], 
                       transform=ccrs.PlateCarree(),
                       c=colors[i], s=120, alpha=0.8, edgecolor='black', linewidth=1,
                       label=dist_name)
    
    ax.set_extent([44, 64, 25, 40], crs=ccrs.PlateCarree())
    ax.set_title('نقشه توزیع بهترین مدل (با Cartopy)', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Cartopy در دسترس نیست. برای مشاهده نقشه حرفه‌ای، Cartopy را نصب کنید.")

In [ ]:
# ============================================================================
# ۵. نمودارهای تعاملی با Plotly (اختیاری)
# ============================================================================

if plotly_available:
    import plotly.express as px
    
    fig = px.scatter_mapbox(
        df_map,
        lat='lat',
        lon='lon',
        color='distribution',
        hover_data={'code': True},
        color_discrete_sequence=px.colors.qualitative.Set1,
        zoom=5,
        height=600,
        title='نقشه تعاملی توزیع‌ها'
    )
    fig.update_layout(mapbox_style='carto-positron')
    fig.show()
else:
    print("⚠️ Plotly در دسترس نیست. برای نقشه تعاملی، Plotly را نصب کنید.")

In [ ]:
# ============================================================================
# ۶. نمودار جعبه‌ای (Boxplot) برای مقایسه مدل‌ها
# ============================================================================

# جمع‌آوری AICc مدل‌ها
model_names = []
aicc_values = []
for name, res in results.items():
    if 'aicc' in res and not np.isnan(res['aicc']):
        model_names.append(name)
        aicc_values.append(res['aicc'])

# ایجاد DataFrame
df_aicc = pd.DataFrame({'مدل': model_names, 'AICc': aicc_values})

fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(data=df_aicc, x='مدل', y='AICc', ax=ax, palette='Set2')
ax.set_title('مقایسه AICc مدل‌های مختلف', fontsize=14, fontweight='bold')
ax.set_xlabel('مدل', fontsize=12)
ax.set_ylabel('AICc', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# ۷. ذخیره نمودارها
# ============================================================================

output_dir = os.path.join(project_root, 'visualizations')
os.makedirs(output_dir, exist_ok=True)

print(f"📁 پوشه {output_dir} برای ذخیره نمودارها ایجاد شد.")

# ذخیره یک نمودار نمونه
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(data_year, color='blue', alpha=0.7, linewidth=1.5)
ax.set_xlabel('روز سال')
ax.set_ylabel('دما (°C)')
ax.set_title('سری زمانی دمای میانگین')
ax.grid(True, alpha=0.3)

plt.savefig(os.path.join(output_dir, 'timeseries.png'), dpi=300, bbox_inches='tight')
plt.close()

print(f"✅ نمودار ذخیره شد: {os.path.join(output_dir, 'timeseries.png')}")

## 📋 جمع‌بندی

در این نوت‌بوک یاد گرفتید:

✅ بارگذاری نتایج از فایل Zarr
✅ رسم سری‌های زمانی
✅ رسم هیستوگرام و منحنی‌های توزیع
✅ رسم QQ-Plot
✅ رسم نقشه‌های مکانی با scatter و Cartopy
✅ رسم نمودارهای تعاملی با Plotly
✅ رسم نمودار جعبه‌ای (Boxplot)
✅ ذخیره نمودارها در فرمت‌های مختلف

---

**نکات کلیدی:**

1. **matplotlib** پایه‌ترین کتابخانه مصورسازی است.
2. **seaborn** برای نمودارهای آماری بسیار مفید است.
3. **Cartopy** برای نقشه‌های جغرافیایی حرفه‌ای استفاده می‌شود.
4. **Plotly** برای نمودارهای تعاملی و وب‌محور مناسب است.
5. همیشه نمودارها را با کیفیت بالا (dpi=300) ذخیره کنید.

---

**مراحل بعدی:**
- نوت‌بوک ۰۹: افزودن توزیع سفارشی
- نوت‌بوک ۱۰: کاربرد پیشرفته